# LangChain 入门笔记本

欢迎！本笔记本带你用最少的代码跑通 LangChain 的核心流程：调用模型 → 提示词模板 → 串成链。

建议按顺序逐个单元格运行（Shift+Enter）。运行前请确保：
- 已激活 `.venv` 虚拟环境
- 已执行 `pip install -r requirements.txt`
- 已复制 `.env.example` 为 `.env` 并填入 `OPENAI_API_KEY`

> 详细概念讲解见 `docs/01-models-and-prompts.md` 与 `docs/02-chains.md`。


## 0. 环境检查

先确认能加载 `.env` 中的 API Key。


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.getenv('OPENAI_API_KEY'):
    raise SystemExit('✗ 未找到 OPENAI_API_KEY，请复制 .env.example 为 .env 并填写。')
print('✓ 环境就绪，API Key 已加载')


## 1. 你的第一个模型调用

`ChatOpenAI` 是对话模型统一的入口。直接调用 `.invoke('...')` 即可得到回复。


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'))
response = llm.invoke('用一句话介绍 LangChain。')
print(response.content)


## 2. 用提示词模板管理输入

硬编码提示词不利于复用。`ChatPromptTemplate` 把可变部分参数化，支持 system / human 多角色消息。


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ('system', '你是一个{role}，请用专业且简洁的语言回答。'),
    ('human', '{question}'),
])

print(prompt.format(role='Python 专家', question='什么是装饰器？'))


## 3. 把「提示词 → 模型 → 解析」串成一条链（LCEL）

用管道符 `|` 把组件串起来，这便是 LangChain Expression Language（LCEL）。整条链本身也是一个可调用对象。


In [ ]:
chain = prompt | llm | StrOutputParser()
print(chain.invoke({'role': 'Python 专家', 'question': '什么是装饰器？'}))


## 4. 动手改参数

把上面的 `role` 或 `question` 换成别的值再运行，体会模板的复用性。


In [ ]:
chain.invoke({'role': '历史老师', 'question': '简述第一次世界大战的起因。'})


## 5. 下一步

- 多轮对话与记忆：运行 `examples/03_memory.py`，参见 `docs/03-memory.md`
- 检索增强（RAG）：运行 `examples/04_rag.py`，参见 `docs/04-retrieval-and-rag.md`
- 智能代理：运行 `examples/05_agents.py`，参见 `docs/05-agents.md`

完成后，你可以在 `notebooks/` 中新建自己的笔记本继续练习。
